# IP certificates in the wild

In [1]:
# Some imports
import plotly.graph_objects as go
import collections
import ipaddress
import glob
import json

In [2]:
# Target protocols

port_protocol = {21: 'FTP', 25: 'SMTP', 110: 'POP3', 143: 'IMAP', 443: 'HTTPS/DoH', 465: 'SMTPS', 587: 'Submission', 636: 'LDAPS', 853: 'DoT/DoQ', 990: 'FTPS', 993: 'IMAPS', 995: 'POP3S', 1433: 'MSSQL', 2376: 'Docker', 3306: 'MySQL', 3389: 'RDP', 5432: 'PostgreSQL', 5671: 'AMQPS', 6443: 'Kubernetes', 7547: 'CWMP', 8443: 'HTTPS', 8883: 'MQTTS', 9200: 'Elasticsearch', 27017: 'MongoDB'}

## Certstream dataset

In [3]:
# Read the dataset and get some stats

stream_all =0
stream_ips, stream_issuers = set(), list()
serials = set()

for scan in glob.glob(f"../data/deployment/certstream/*.json"):
    with open(scan, "r") as f:
        for line in f:
            certificate = json.loads(line)
            # Update the certifiate counter
            stream_all += 1
            # Do not process each certificate more than once
            serial = certificate["serial_number"]
            if serial not in serials:
                # IPs
                san_ips = [i[3:] for i in certificate["extensions"]["subjectAltName"].split(", ") if i.startswith("IP:")]
                for ip in san_ips:
                    stream_ips.add(ip)
                # Issuers
                stream_issuers.append(certificate["issuer"]["O"])
                # Store the serial
                serials.add(serial)

In [4]:
# Print some stats
ips_v4 = len([i for i in set(stream_ips) if ":" not in i])
ips_v6 = len([i for i in set(stream_ips) if ":" in i])

print(f"All measurements: {stream_all:,}")
print(f"  Unique certificates: {len(serials):,}")
print(f"    Unique IPs: {len(set(stream_ips)):,} ({ips_v4:,} IPv4 and {ips_v6:,} IPv6)")
print(f"    Unique issuers: {len(set(stream_issuers)):,}")
print(f"-----------------------")
print(f"Issuers:")
for issuer,count in collections.Counter(stream_issuers).most_common():
    print(f"  {issuer}: {count:,}")

All measurements: 1,480,863
  Unique certificates: 371,980
    Unique IPs: 295,759 (295,759 IPv4 and 0 IPv6)
    Unique issuers: 18
-----------------------
Issuers:
  Let's Encrypt: 369,031
  ZeroSSL GmbH: 2,824
  Sectigo Limited: 95
  sslTrus Inc.: 4
  GoGetSSL: 4
  UniTrust: 4
  Asseco Data Systems S.A.: 3
  GlobalSign nv-sa: 2
  WoTrus CA Limited: 2
  ZoTrus Technology Limited: 2
  JoySSL Limited: 2
  sslTrus: 1
  Entrust Limited: 1
  Google Trust Services: 1
  Unizeto Technologies S.A.: 1
  SSL Corporation: 1
  DigiCert Inc: 1
  ZeroSSL: 1


In [5]:
# The above as a Latex table
rank = 1 
for issuer,count in collections.Counter(stream_issuers).most_common():
    print(f"{rank}. & {issuer} & {count:,} & {round(count*100/len(stream_issuers),2)}\\% \\\\")
    rank += 1

1. & Let's Encrypt & 369,031 & 99.21\% \\
2. & ZeroSSL GmbH & 2,824 & 0.76\% \\
3. & Sectigo Limited & 95 & 0.03\% \\
4. & sslTrus Inc. & 4 & 0.0\% \\
5. & GoGetSSL & 4 & 0.0\% \\
6. & UniTrust & 4 & 0.0\% \\
7. & Asseco Data Systems S.A. & 3 & 0.0\% \\
8. & GlobalSign nv-sa & 2 & 0.0\% \\
9. & WoTrus CA Limited & 2 & 0.0\% \\
10. & ZoTrus Technology Limited & 2 & 0.0\% \\
11. & JoySSL Limited & 2 & 0.0\% \\
12. & sslTrus & 1 & 0.0\% \\
13. & Entrust Limited & 1 & 0.0\% \\
14. & Google Trust Services & 1 & 0.0\% \\
15. & Unizeto Technologies S.A. & 1 & 0.0\% \\
16. & SSL Corporation & 1 & 0.0\% \\
17. & DigiCert Inc & 1 & 0.0\% \\
18. & ZeroSSL & 1 & 0.0\% \\


## Active probing

In [6]:
# Read the dataset and get some stats

probing_ips_all, probing_ips_successful, probing_ips_certs, probing_ips_stream = set(), set(), set(), set()
probing_ports_successful, probing_ports_certs, probing_ports_stream = list(), list(), list()
serials_overlap, observations = set(), set()

def san_has_ips(san_entries):
    """Check if a SAN list contains IPs"""

    has_ips = False
    for entry in san_entries:
        try:
            ipaddress.ip_address(entry)
            has_ips = True
            break
        except:
            pass

    return has_ips

for scan in glob.glob(f"../data/deployment/probing/*.json"):
    with open(scan, "r") as f:
        for line in f:
            scan = json.loads(line)
            # Focus on successful measurements
            if scan["success"]:
                # Ports and IPs on which we successfully retrieved certificates
                probing_ports_successful.append(scan["port"])
                probing_ips_successful.add(scan["ip"])
                # Focus on measurements with IP certificates
                if san_has_ips(san_entries=scan["san"]):
                    probing_ips_certs.add(scan["ip"])
                    probing_ports_certs.append(scan["port"])
                    # Focus on measurements with certs observed in the certstream
                    if scan["serial"] in serials:
                        probing_ips_stream.add(scan["ip"])
                        probing_ports_stream.append(scan["port"])
                        serials_overlap.add(scan["serial"])
                        # Log each unique observation
                        observations.add((scan["serial"],scan["ip"],scan["port"],scan["issuer"]))
            # Update counters
            probing_ips_all.add(scan["ip"])

In [7]:
# Print some stats
print(f"All IPs: {len(set(stream_ips)):,}")
print(f"All ports: 24")
print(f"  -------------------------")
print(f"  All IPs measured: {len(probing_ips_all):,}")
print(f"  All ports measued: 24")
print(f"    -----------------------")
print(f"    Successful IPs: {len(probing_ips_successful):,}")
print(f"    Ports successful: {len(set(probing_ports_successful))}")
print(f"      -------------------------")
print(f"      IPs with IP certs: {len(probing_ips_certs):,}")
print(f"       Ports with IP certs: {len(set(probing_ports_certs))}")
print(f"        -----------------------------------------")
print(f"        IPs with IP certs from the stream: {len(probing_ips_stream):,}")
print(f"        Ports with IP certs from the stream: {len(set(probing_ports_stream))}")
print(f"          --------------------------------")
print(f"          Overlapping certificates: {len(serials_overlap):,}")

All IPs: 295,759
All ports: 24
  -------------------------
  All IPs measured: 279,153
  All ports measued: 24
    -----------------------
    Successful IPs: 161,375
    Ports successful: 24
      -------------------------
      IPs with IP certs: 44,059
       Ports with IP certs: 22
        -----------------------------------------
        IPs with IP certs from the stream: 37,867
        Ports with IP certs from the stream: 21
          --------------------------------
          Overlapping certificates: 29,656


In [8]:
layout = go.Layout(
    autosize=False,
    margin = {'l':0,'r':0,'t':0},
    plot_bgcolor='#F5F5F5',
    width=900,
    height=350,
)

fig = go.Figure(go.Waterfall(
    orientation = "v",
    measure = ["relative", "relative", "relative", "relative", "relative", "total"],
    increasing={"marker": {"color": "#6BAA75"}},
    decreasing={"marker": {"color": "#C96A6A"}},
    totals={"marker": {"color": "#3A506B"}},
    x = [
        "&#9312; All the IPs<br> seen in the stream", 
        "&#9313; Certstream hosts<br> with certificates valid <br>during the scan", 
        "&#9314; Probed hosts with<br> TLS certificates", 
        "&#9315; Probed hosts with<br> IP TLS certificates", 
        "&#9316; Probed hosts with<br> IP TLS certificates<br> seen in the stream", 
        "&#9317; Total hosts"
        ],
    textposition = "outside",
    text = [
        f"{len(set(stream_ips)):,}", 
        f"{-(len(set(stream_ips))-len(probing_ips_all)):,}", 
        f"{-(len(probing_ips_all)-len(probing_ips_successful)):,}",
        f"{-(len(probing_ips_successful)-len(probing_ips_certs)):,}", 
        f"{-(len(probing_ips_certs)-len(probing_ips_stream)):,}",
        f"{len(probing_ips_stream):,}"
        ],
    y = [
        len(set(stream_ips)), 
        -(len(set(stream_ips))-len(probing_ips_all)), 
        -(len(probing_ips_all)-len(probing_ips_successful)), 
        -(len(probing_ips_successful)-len(probing_ips_certs)), 
        -(len(probing_ips_certs)-len(probing_ips_stream)),
        len(probing_ips_stream),
        None
        ],
    connector = {"line":{"color":"rgb(63, 63, 63)"}},
),
layout=layout)

fig.update_layout(yaxis_range=[-100000,400000], yaxis = dict(tickmode='array',tickvals = [-100000,0,100000,200000,300000,400000], ticktext=["", "0", "100k", "200k", "300k", "400k"]))
fig.update_xaxes(tickangle=0)
fig.show()
fig.write_image("../data/figures/waterfall_ips.pdf")

## Overlapped certificates

In [9]:
observed_serials = [i[0] for i in observations]
observed_ports = [i[2] for i in observations]
observed_issuers = [i[3].split(",")[1].split("=")[1] for i in observations]

print(f"All observations: {len(observations):,}")
print(f"  Serials: {len(set(observed_serials)):,}")
print(f"  Port: {len(set(observed_ports)):,}")
print(f"  Issuers: {len(set(observed_issuers)):,}")

All observations: 42,755
  Serials: 29,656
  Port: 21
  Issuers: 12


In [10]:
# Find the port distribution
for port,count in collections.Counter(observed_ports).most_common():
    print(f"{port} ({port_protocol[port]}): {count:,} ({round(count*100/len(observations),2)}%)")

443 (HTTPS/DoH): 38,951 (91.1%)
8443 (HTTPS): 3,458 (8.09%)
853 (DoT/DoQ): 70 (0.16%)
5432 (PostgreSQL): 52 (0.12%)
3389 (RDP): 50 (0.12%)
8883 (MQTTS): 40 (0.09%)
6443 (Kubernetes): 33 (0.08%)
993 (IMAPS): 22 (0.05%)
465 (SMTPS): 16 (0.04%)
3306 (MySQL): 15 (0.04%)
9200 (Elasticsearch): 13 (0.03%)
995 (POP3S): 8 (0.02%)
1433 (MSSQL): 7 (0.02%)
27017 (MongoDB): 6 (0.01%)
2376 (Docker): 4 (0.01%)
990 (FTPS): 3 (0.01%)
5671 (AMQPS): 2 (0.0%)
21 (FTP): 2 (0.0%)
143 (IMAP): 1 (0.0%)
587 (Submission): 1 (0.0%)
25 (SMTP): 1 (0.0%)


In [11]:
# The above as a Latex table
counter = 1
for port,count in collections.Counter(observed_ports).most_common():
    print(f"{counter}. & {port} & {port_protocol[port]} & {count:,} & {round(count*100/len(observations),2)}\\% \\\\")
    counter += 1

1. & 443 & HTTPS/DoH & 38,951 & 91.1\% \\
2. & 8443 & HTTPS & 3,458 & 8.09\% \\
3. & 853 & DoT/DoQ & 70 & 0.16\% \\
4. & 5432 & PostgreSQL & 52 & 0.12\% \\
5. & 3389 & RDP & 50 & 0.12\% \\
6. & 8883 & MQTTS & 40 & 0.09\% \\
7. & 6443 & Kubernetes & 33 & 0.08\% \\
8. & 993 & IMAPS & 22 & 0.05\% \\
9. & 465 & SMTPS & 16 & 0.04\% \\
10. & 3306 & MySQL & 15 & 0.04\% \\
11. & 9200 & Elasticsearch & 13 & 0.03\% \\
12. & 995 & POP3S & 8 & 0.02\% \\
13. & 1433 & MSSQL & 7 & 0.02\% \\
14. & 27017 & MongoDB & 6 & 0.01\% \\
15. & 2376 & Docker & 4 & 0.01\% \\
16. & 990 & FTPS & 3 & 0.01\% \\
17. & 5671 & AMQPS & 2 & 0.0\% \\
18. & 21 & FTP & 2 & 0.0\% \\
19. & 143 & IMAP & 1 & 0.0\% \\
20. & 587 & Submission & 1 & 0.0\% \\
21. & 25 & SMTP & 1 & 0.0\% \\


In [12]:
# Find the port distribution
for issuer,count in collections.Counter(observed_issuers).most_common():
    print(f"{issuer}: {count:,} ({round(count*100/len(observations),2)}%)")

Let's Encrypt: 35,915 (84.0%)
ZeroSSL GmbH: 6,772 (15.84%)
Sectigo Limited: 45 (0.11%)
Asseco Data Systems S.A.: 9 (0.02%)
GoGetSSL: 4 (0.01%)
sslTrus Inc.: 3 (0.01%)
ZoTrus Technology Limited: 2 (0.0%)
JoySSL Limited: 1 (0.0%)
GlobalSign nv-sa: 1 (0.0%)
SSL Corporation: 1 (0.0%)
www.digicert.com: 1 (0.0%)
WoTrus CA Limited: 1 (0.0%)


In [13]:
# The above as a Latex table
counter = 1
for issuer,count in collections.Counter(observed_issuers).most_common():
    print(f"{counter}. & {issuer} & {count:,} & {round(count*100/len(observations),2)}\\% \\\\")
    counter += 1

1. & Let's Encrypt & 35,915 & 84.0\% \\
2. & ZeroSSL GmbH & 6,772 & 15.84\% \\
3. & Sectigo Limited & 45 & 0.11\% \\
4. & Asseco Data Systems S.A. & 9 & 0.02\% \\
5. & GoGetSSL & 4 & 0.01\% \\
6. & sslTrus Inc. & 3 & 0.01\% \\
7. & ZoTrus Technology Limited & 2 & 0.0\% \\
8. & JoySSL Limited & 1 & 0.0\% \\
9. & GlobalSign nv-sa & 1 & 0.0\% \\
10. & SSL Corporation & 1 & 0.0\% \\
11. & www.digicert.com & 1 & 0.0\% \\
12. & WoTrus CA Limited & 1 & 0.0\% \\
